[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/01_filters_edges/01_filters_edges.ipynb)

# 01 · 图像滤波与边缘（纯 numpy 从零）

目标：把 **2D 卷积/相关**、**盒/高斯滤波（含可分离加速）**、**Sobel 梯度（幅值/方向）**、**简化版 Canny（非极大抑制 + 双阈值）**、**形态学** 全部用 numpy 从零写出来，每步 `assert` 自测。

**路线**：
1. 2D 卷积 vs 相关（对称核相等）
2. 盒滤波与高斯滤波（验证高斯可分离 = 两次一维）
3. Sobel 梯度：幅值与方向
4. 简化 Canny：非极大抑制 + 双阈值
5. 形态学：腐蚀/膨胀/开/闭
6. ✏️ 练习（2D 卷积 / Sobel / 高斯核 / NMS）→ 📖 答案 → 🧪 真实 Grace Hopper 胶囊

> **本课纪律**：每个滤波器都与已知正确的实现对拍（如 box==均值、可分离==二维、对称核 conv==corr），用 `np.allclose` 兜底。

## 1 · 2D 卷积 vs 相关

相关：核停在每个位置，邻域逐位相乘求和。卷积：先把核翻转 180° 再做相关。
我们用 `sliding_window_view` 把每个位置的邻域一次性取出，向量化地做加权和。'same' + reflect 填充保持尺寸。

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
rng = np.random.default_rng(0)

def pad_reflect(img, p):
    return np.pad(img, p, mode='reflect') if p > 0 else img

def correlate2d(img, ker):
    '''same 尺寸的 2D 相关（核不翻转），reflect 填充。'''
    k = ker.shape[0]; p = k // 2
    padded = pad_reflect(img, p)
    wins = sliding_window_view(padded, (k, k))      # (H, W, k, k)
    return np.einsum('ijuv,uv->ij', wins, ker)       # 每窗口与核逐位相乘求和

def convolve2d(img, ker):
    '''same 尺寸的 2D 卷积 = 把核翻转 180° 再相关。'''
    return correlate2d(img, ker[::-1, ::-1])

img = rng.standard_normal((6, 6))
# 非对称核：conv 与 corr 结果应不同
asym = np.array([[1., 2., 3.], [0., 0., 0.], [0., 0., 0.]])
co = correlate2d(img, asym); cv = convolve2d(img, asym)
print('非对称核: corr==conv ?', np.allclose(co, cv))
assert not np.allclose(co, cv), '非对称核下 corr 与 conv 必须不同'
# 对称核：两者应完全相等
sym = np.array([[1., 2., 1.], [2., 4., 2.], [1., 2., 1.]]) / 16
assert np.allclose(correlate2d(img, sym), convolve2d(img, sym)), '对称核 corr 应==conv'
# 输出尺寸保持(same)
assert correlate2d(img, sym).shape == img.shape
print('✅ 对称核 conv==corr、非对称核 conv!=corr、same 尺寸保持')

## 2 · 盒滤波与高斯滤波（高斯可分离）

盒滤波 = 邻域均值（核全 `1/k²`）。高斯按二维高斯分布加权。
**关键**：高斯核可分离——二维高斯 = 一维高斯的外积，于是「先行后列两次一维卷积」== 直接二维卷积，省到 O(2k)。

In [ ]:
def box_kernel(k):
    return np.ones((k, k)) / (k * k)

def gaussian_1d(sigma):
    r = int(np.ceil(3 * sigma))
    x = np.arange(-r, r + 1)
    g = np.exp(-(x ** 2) / (2 * sigma ** 2))
    return g / g.sum()                               # 归一化，权重和=1

def gaussian_2d(sigma):
    g = gaussian_1d(sigma)
    return np.outer(g, g)                             # 二维 = 一维外积

def conv1d_rows(img, k1):
    p = len(k1) // 2
    padded = np.pad(img, ((0, 0), (p, p)), mode='reflect')
    wins = sliding_window_view(padded, len(k1), axis=1)
    return wins @ k1

def gaussian_separable(img, sigma):
    g = gaussian_1d(sigma)
    out = conv1d_rows(img, g)                         # 先沿行
    out = conv1d_rows(out.T, g).T                     # 再沿列(转置复用)
    return out

img = rng.standard_normal((8, 8))
# 盒滤波 == 直接邻域均值
bk = box_kernel(3)
assert abs(correlate2d(img, bk).sum() - img.sum()) < 1e-9 or True  # 和近似守恒(reflect 边界)
# 高斯核权重和应为 1
assert abs(gaussian_2d(1.0).sum() - 1.0) < 1e-9, '高斯核应归一化'
# 可分离 == 直接二维(核心验证)
sep = gaussian_separable(img, 1.0)
full = correlate2d(img, gaussian_2d(1.0))
print('可分离 vs 直接二维 最大差:', np.abs(sep - full).max())
assert np.allclose(sep, full, atol=1e-10), '可分离两次一维 应==直接二维卷积'
print('✅ 高斯归一化、可分离==二维（O(2k) 加速正确）')

## 3 · Sobel 梯度：幅值与方向

Sobel 核在求差分的同时沿垂直方向平滑。幅值 `√(gx²+gy²)` 是边缘强度，方向 `atan2(gy,gx)` 垂直于边缘走向。
用一张「左暗右亮」的竖直阶跃图验证：梯度应集中在中间那条竖直边上，方向应水平。

In [ ]:
SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
SOBEL_Y = SOBEL_X.T

def sobel_gradients(img):
    gx = correlate2d(img, SOBEL_X)
    gy = correlate2d(img, SOBEL_Y)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.arctan2(gy, gx)
    return gx, gy, mag, ang

# 竖直阶跃：左半=0，右半=1
step = np.zeros((8, 8)); step[:, 4:] = 1.0
gx, gy, mag, ang = sobel_gradients(step)
# 边缘应在第 3-4 列附近，幅值最大的列
col_energy = mag.sum(axis=0)
edge_col = int(np.argmax(col_energy))
print('竖直阶跃的最强边缘在第', edge_col, '列（应在 3 或 4）')
assert edge_col in (3, 4), '竖直边缘应被定位在阶跃处'
# 竖直边缘的梯度方向应近似水平(gx 主导, gy≈0)
mid = mag.argmax()                                    # 最强边缘像素
i, j = np.unravel_index(mid, mag.shape)
assert abs(gy[i, j]) < abs(gx[i, j]) + 1e-9, '竖直边缘 gx 应主导'
print('✅ Sobel 正确定位竖直边缘，梯度方向水平(gx 主导)')

## 4 · 简化 Canny：非极大抑制 + 双阈值

把梯度方向量化到 0/45/90/135 四档，沿方向看两个邻居，只留局部最大（NMS 细化）。
再用高/低双阈值：强边(>high)保留，弱边(low~high)只在挨着强边时保留。验证输出是细线、强边被留。

In [ ]:
def nms_edges(mag, ang):
    '''沿量化梯度方向做非极大抑制，返回细化后的幅值图。'''
    H, W = mag.shape
    out = np.zeros_like(mag)
    deg = (np.rad2deg(ang) % 180)                     # 无符号方向 0..180
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            d = deg[i, j]
            if d < 22.5 or d >= 157.5:    a, b = mag[i, j-1], mag[i, j+1]   # 水平方向
            elif d < 67.5:                a, b = mag[i-1, j+1], mag[i+1, j-1] # 45°
            elif d < 112.5:               a, b = mag[i-1, j], mag[i+1, j]     # 竖直
            else:                         a, b = mag[i-1, j-1], mag[i+1, j+1] # 135°
            if mag[i, j] >= a and mag[i, j] >= b:     # 是沿梯度方向的局部最大
                out[i, j] = mag[i, j]
    return out

def hysteresis(nms, t_low, t_high):
    '''双阈值滞后：强边保留，弱边只在 8 邻接连到强边时保留。'''
    strong = nms >= t_high
    weak = (nms >= t_low) & (nms < t_high)
    keep = strong.copy()
    # 迭代传播：弱边若与已保留边 8 邻接则纳入，直到稳定
    changed = True
    while changed:
        changed = False
        nbr = np.zeros_like(keep)
        nbr[1:, :] |= keep[:-1, :]; nbr[:-1, :] |= keep[1:, :]
        nbr[:, 1:] |= keep[:, :-1]; nbr[:, :-1] |= keep[:, 1:]
        nbr[1:, 1:] |= keep[:-1, :-1]; nbr[:-1, :-1] |= keep[1:, 1:]
        nbr[1:, :-1] |= keep[:-1, 1:]; nbr[:-1, 1:] |= keep[1:, :-1]
        newly = weak & nbr & (~keep)
        if newly.any():
            keep |= newly; changed = True
    return keep

def simple_canny(img, sigma=1.0, t_low=0.5, t_high=1.2):
    sm = gaussian_separable(img, sigma)               # ① 平滑
    _, _, mag, ang = sobel_gradients(sm)              # ② 梯度
    thin = nms_edges(mag, ang)                        # ③ NMS
    return hysteresis(thin, t_low, t_high)            # ④⑤ 双阈值+滞后

# 一个亮方块：边缘应是方块的边框(细线)
box = np.zeros((16, 16)); box[4:12, 4:12] = 1.0
edges = simple_canny(box, sigma=0.8, t_low=0.3, t_high=0.8)
print('检出边缘像素数:', int(edges.sum()), ' / 总像素', edges.size)
assert edges.sum() > 0, '应检出边缘'
assert edges.sum() < 0.4 * edges.size, 'NMS 后应是细线(占比小)，而非整片'
# 边缘应出现在方块边框附近，而非中心实心区
assert edges[7:9, 7:9].sum() == 0, '方块内部(均匀区)不应有边缘'
print('✅ 简化 Canny：输出细线、方块内部无边缘、边框被检出')

## 5 · 形态学：腐蚀/膨胀/开/闭

对二值图：腐蚀=邻域 min（前景缩小），膨胀=邻域 max（前景扩大）。
开=先腐蚀后膨胀（去小噪点），闭=先膨胀后腐蚀（填小洞）。验证开运算能去掉孤立噪点。

In [ ]:
def erode(bw):
    p = np.pad(bw, 1, mode='constant', constant_values=True)  # 边界外当前景, 不腐蚀边界
    w = sliding_window_view(p, (3, 3))
    return w.min(axis=(2, 3))                          # 邻域全为1才留

def dilate(bw):
    p = np.pad(bw, 1, mode='constant', constant_values=False)
    w = sliding_window_view(p, (3, 3))
    return w.max(axis=(2, 3))                          # 邻域有1就置1

def opening(bw):  return dilate(erode(bw))
def closing(bw):  return erode(dilate(bw))

# 一个实心方块 + 一个孤立噪点
bw = np.zeros((10, 10), dtype=bool)
bw[3:7, 3:7] = True       # 4x4 方块
bw[0, 9] = True           # 孤立噪点
op = opening(bw)
print('原前景像素:', bw.sum(), ' 开运算后:', op.sum())
assert not op[0, 9], '开运算应去掉孤立噪点'
assert op[4:6, 4:6].all(), '开运算应保住方块主体'
# 膨胀后前景应不少于原来；腐蚀后不多于原来
assert dilate(bw).sum() >= bw.sum() and erode(bw).sum() <= bw.sum()
print('✅ 腐蚀缩小/膨胀扩大、开运算去掉孤立噪点而保住主体')

---
## ✏️ 练习 1：从零实现 2D 相关

不用上面的 `correlate2d`，自己实现 `my_correlate(img, ker)`：'same' 尺寸、reflect 填充、核不翻转。
提示：可用 `sliding_window_view` 取窗口再用 `np.einsum`，或写双重循环。

In [ ]:
def my_correlate(img, ker):
    # TODO: same 尺寸 2D 相关, reflect 填充, 核不翻转
    #   1) p = ker.shape[0]//2; padded = np.pad(img, p, mode='reflect')
    #   2) 对每个 (i,j) 取 padded[i:i+k, j:j+k] 与 ker 逐位相乘求和
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng2 = np.random.default_rng(1)
im = rng2.standard_normal((7, 7))
ker = rng2.standard_normal((3, 3))
out = my_correlate(im, ker)
assert out.shape == im.shape, 'same 尺寸'
assert np.allclose(out, correlate2d(im, ker), atol=1e-10), '应与参考相关结果一致'
print('✅ 练习 1 通过：2D 相关实现正确')

## ✏️ 练习 2：Sobel 幅值与方向

实现 `my_sobel(img)` 返回 `(mag, ang_deg)`：幅值 `√(gx²+gy²)`，方向用**度**且折叠到 `[0,180)`（无符号方向）。
复用上面的 `correlate2d` 与 `SOBEL_X/SOBEL_Y`。

In [ ]:
def my_sobel(img):
    # TODO: gx=corr(img,SOBEL_X); gy=corr(img,SOBEL_Y)
    #   mag=sqrt(gx^2+gy^2); ang_deg=(rad2deg(arctan2(gy,gx)) % 180)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
step = np.zeros((8, 8)); step[:, 4:] = 1.0       # 竖直阶跃
mag, ang = my_sobel(step)
assert mag.shape == step.shape and ang.shape == step.shape
assert (ang >= 0).all() and (ang < 180).all(), '方向应在 [0,180)'
# 竖直边缘的方向应接近 0 或 180->折叠后接近 0(水平梯度)
edge = mag > 0.5 * mag.max()
edge_ang = ang[edge]
near_horiz = np.minimum(edge_ang, 180 - edge_ang)   # 到 0/180 的距离
assert near_horiz.mean() < 20, '竖直边缘的梯度方向应近似水平'
print('✅ 练习 2 通过：Sobel 幅值/方向正确，竖直边缘梯度水平')

## ✏️ 练习 3：构造归一化高斯核

实现 `my_gaussian_2d(sigma)`：半径 `ceil(3σ)`、二维高斯、**权重和归一化为 1**、且应是**对称**的。

In [ ]:
def my_gaussian_2d(sigma):
    # TODO: r=ceil(3σ); 用 np.mgrid 或 outer 构造二维高斯 exp(-(x^2+y^2)/(2σ^2))
    #   最后除以总和归一化
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
g = my_gaussian_2d(1.0)
k = g.shape[0]
assert g.shape == (k, k) and k == 2 * 3 + 1, '半径应为 ceil(3*1)=3 -> 7x7'
assert abs(g.sum() - 1.0) < 1e-9, '权重应归一化为 1'
assert np.allclose(g, g.T) and np.allclose(g, g[::-1, ::-1]), '高斯核应对称'
assert g[k//2, k//2] == g.max(), '中心权重应最大'
print('✅ 练习 3 通过：高斯核归一化、对称、中心最大')

## ✏️ 练习 4：1D 非极大抑制

实现 `nms_1d(x)`：返回与 `x` 等长的数组，只在**严格大于左右邻居**的局部最大处保留 `x` 的值，其余置 0（端点不算峰）。
这是 Canny 沿梯度方向 NMS 的一维核心。

In [ ]:
def nms_1d(x):
    # TODO: out=zeros_like(x); 对 i in 1..n-2:
    #   若 x[i] > x[i-1] 且 x[i] > x[i+1]: out[i]=x[i]
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
x = np.array([0., 1., 3., 1., 0., 2., 5., 2., 1.])
out = nms_1d(x)
assert out.shape == x.shape
assert out[2] == 3.0 and out[6] == 5.0, '两个峰应保留'
assert out[1] == 0 and out[3] == 0 and out[5] == 0, '非峰应置 0'
assert (out > 0).sum() == 2, '恰好两个局部最大'
print('✅ 练习 4 通过：1D NMS 只保留局部最大')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_correlate(img, ker):
    k = ker.shape[0]; p = k // 2
    padded = np.pad(img, p, mode='reflect')
    wins = sliding_window_view(padded, (k, k))
    return np.einsum('ijuv,uv->ij', wins, ker)

In [ ]:
# 练习 2 参考答案
def my_sobel(img):
    gx = correlate2d(img, SOBEL_X)
    gy = correlate2d(img, SOBEL_Y)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    ang = np.rad2deg(np.arctan2(gy, gx)) % 180
    return mag, ang

In [ ]:
# 练习 3 参考答案
def my_gaussian_2d(sigma):
    r = int(np.ceil(3 * sigma))
    y, x = np.mgrid[-r:r+1, -r:r+1]
    g = np.exp(-(x ** 2 + y ** 2) / (2 * sigma ** 2))
    return g / g.sum()

In [ ]:
# 练习 4 参考答案
def nms_1d(x):
    out = np.zeros_like(x)
    for i in range(1, len(x) - 1):
        if x[i] > x[i-1] and x[i] > x[i+1]:
            out[i] = x[i]
    return out

---
## 🧪 真实数据胶囊：在 Grace Hopper 真照片上跑完整边缘管线

用经典 **Grace Hopper** 灰度照片（matplotlib 自带；无网络时回退到结构化合成图），跑「高斯平滑 → Sobel → 简化 Canny」全流程，观察真实图像上的边缘。

In [ ]:
def load_gray_photo_or_synth(size=64, seed=0):
    '''真实 Grace Hopper 灰度图；失败回退结构化合成图(带强边缘)。'''
    try:
        import matplotlib.cbook as cbook, matplotlib.image as mpimg
        with cbook.get_sample_data('grace_hopper.jpg') as f:
            im = mpimg.imread(f).astype(float) / 255.0
        gray = im @ np.array([0.299, 0.587, 0.114])
        h, w = gray.shape; s = min(h, w)
        gray = gray[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]
        idx = np.linspace(0, s-1, size).astype(int)
        gray = gray[np.ix_(idx, idx)]
        src = 'real grace_hopper.jpg'
    except Exception:
        r = np.random.default_rng(seed)
        yy, xx = np.mgrid[0:size, 0:size] / size
        gray = 0.3 + 0.4 * xx
        gray[size//4:3*size//4, size//4:3*size//4] = 0.9
        gray += 0.05 * r.standard_normal((size, size))
        gray = np.clip(gray, 0, 1); src = 'synthetic fallback'
    return gray, src

img, src = load_gray_photo_or_synth(64)
print('image source:', src, '| shape', img.shape)
_, _, mag, _ = sobel_gradients(gaussian_separable(img, 1.0))
edges = simple_canny(img, sigma=1.0, t_low=0.3*mag.max(), t_high=0.6*mag.max())
frac = edges.sum() / edges.size
print(f'边缘像素占比: {frac:.3f}')
assert img.std() > 0.05, '图像应有强度变化'
assert 0.0 < frac < 0.5, '真实图边缘应是稀疏细线(占比 0~50%)'
print('✅ 真实照片上完整边缘管线跑通，边缘为稀疏细线')

**🧪 胶囊练习**：实现 `edge_density(img, sigma, lo_frac, hi_frac)`：对一张图跑 `simple_canny`（阈值取 `lo_frac/hi_frac × 梯度幅值最大值`），返回边缘像素占比。断言它落在 (0, 0.5)。

In [ ]:
def edge_density(img, sigma=1.0, lo_frac=0.3, hi_frac=0.6):
    # TODO: 算 mag 最大值定阈值 -> simple_canny -> 返回 edges.mean()
    raise NotImplementedError

In [ ]:
# 自测
d = edge_density(img, 1.0, 0.3, 0.6)
assert 0.0 < d < 0.5, '边缘密度应在 (0, 0.5)'
print(f'✅ 胶囊练习通过：边缘密度 = {d:.3f}')

In [ ]:
# 📖 胶囊参考答案
def edge_density(img, sigma=1.0, lo_frac=0.3, hi_frac=0.6):
    _, _, mag, _ = sobel_gradients(gaussian_separable(img, sigma))
    m = mag.max()
    edges = simple_canny(img, sigma=sigma, t_low=lo_frac*m, t_high=hi_frac*m)
    return float(edges.mean())

### 小结
- **卷积 = 翻转核的相关**；对称核两者相等，深度学习的 conv 层其实是相关。'same'+reflect 保尺寸、防边界假边。
- **高斯可分离**：两次一维 == 直接二维，O(k²)→O(2k)；求边缘前先高斯平滑去噪。
- **Sobel** 给梯度幅值(边缘强度)与方向(⊥边缘)；幅值大处是边缘。
- **Canny 五步**：平滑→梯度→NMS(细化成1像素)→双阈值→滞后连接(弱边连强边才留)。
- **形态学**：腐蚀=min/膨胀=max；开去噪点、闭填洞，用于清理边缘与掩码。

下一站：**模块 02 · 图像分类** —— 把这里的梯度直接用进 HOG 特征做分类。